# AURORA OMEGA Colab Max

Clean runbook for training the AURORA Omega valuation model from the cached Drive panel.

Run top to bottom. The notebook does not store API keys. It will first try to reuse:

- `/content/drive/MyDrive/blsprime_aurora_omega/panel/featured_panel_autodiscover_2005_2024_1500.parquet`
- `/content/drive/MyDrive/blsprime_aurora_omega/panel/panel_autodiscover_2005_2024_1500.parquet`

If those files exist, no FMP or yfinance download is needed.


## 1. Runtime and Setup

Use a GPU runtime when possible. For a quick smoke run, set `EPOCHS = 3`. For the real run, use 80-160 epochs.


In [ ]:
import os, sys, json, time, random, subprocess
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## 2. Configuration


In [ ]:
REPO_URL = "https://github.com/tbasaure-sys/fin.git"
REPO_REF = "main"

WORKDIR = Path("/content/fin") if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path("/content/drive/MyDrive/blsprime_aurora_omega") if IN_COLAB else Path("./_local_data/blsprime_aurora_omega")
PANEL_ROOT = DRIVE_ROOT / "panel"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"

START_YEAR = 2005
LAST_FEATURE_YEAR = 2024
TRAIN_END_YEAR = 2020
VAL_START_YEAR = 2021

EPOCHS = 80 if torch.cuda.is_available() else 20
BATCH_SIZE = 512 if torch.cuda.is_available() else 256
D_MODEL = 192 if torch.cuda.is_available() else 128
LR = 8e-4
SEED = 7
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

for p in [DRIVE_ROOT, PANEL_ROOT, ARTIFACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("WORKDIR:", WORKDIR)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("Training config:", {"epochs": EPOCHS, "batch": BATCH_SIZE, "d_model": D_MODEL, "device": DEVICE})


## 3. Sync Repo and Imports


In [ ]:
if IN_COLAB:
    if not WORKDIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(WORKDIR)])
    subprocess.check_call(["git", "-C", str(WORKDIR), "fetch", "origin", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "checkout", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", REPO_REF])

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

import scripts.run_aurora_router_local as router
from aurora_omega.data import build_omega_bundle, LENS_NAMES
from aurora_omega.train import TrainConfig, train_omega, evaluate_omega
from aurora_omega.outputs import write_valuation_mri

print("Repo ready:", WORKDIR)
print("router split:", router.TRAIN_END_YEAR, router.VAL_START_YEAR)
print("lenses:", LENS_NAMES)


## 4. Load or Build Featured Panel

This cell is the important fix. It prefers the already-built Drive panel and creates `featured` deterministically. No hidden dependency on previous rescue cells.


In [ ]:
def _first_existing(paths):
    for p in paths:
        if p.exists() and p.stat().st_size > 0:
            return p
    return None

FEATURED_CANDIDATES = [
    PANEL_ROOT / "featured_panel_autodiscover_2005_2024_1500.parquet",
    PANEL_ROOT / "featured_panel_omega_colab_max_2005_2024_1500.parquet",
]

PANEL_CANDIDATES = [
    PANEL_ROOT / "panel_autodiscover_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_cache_only_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_selfcontained_2005_2024_1500.parquet",
]

featured_path = _first_existing(FEATURED_CANDIDATES)
panel_path = _first_existing(PANEL_CANDIDATES)

if featured_path:
    featured = pd.read_parquet(featured_path)
    print("Loaded featured:", featured_path)
elif panel_path:
    panel = pd.read_parquet(panel_path)
    print("Loaded panel:", panel_path, panel.shape, "tickers:", panel["ticker"].nunique())

    featured = router.add_features(panel.copy())
    featured = router.add_lens_predictions(featured)

    featured["omega_regime"] = featured.apply(router.classify_spine_regime, axis=1)
    featured["omega_primary_question"] = featured["omega_regime"].map(router.primary_question_for_regime)

    expectations = featured.apply(router.reverse_dcf_expectations, axis=1)
    featured["omega_expectations_pressure"] = [e.get("valuation_pressure_score", np.nan) for e in expectations]
    featured["omega_feasibility_score"] = [
        router.score_expectation_feasibility(row, e).get("score", np.nan)
        for (_, row), e in zip(featured.iterrows(), expectations)
    ]
    featured["omega_downside_anchor_score"] = [
        router.anchor_lens_checks(row, router.classify_spine_regime(row), e).get("asset_value", {}).get("score", np.nan)
        for (_, row), e in zip(featured.iterrows(), expectations)
    ]

    featured_path = PANEL_ROOT / "featured_panel_autodiscover_2005_2024_1500.parquet"
    featured.to_parquet(featured_path, index=False)
    print("Saved featured:", featured_path)
else:
    raise FileNotFoundError(
        "No panel parquet found. Expected one of: "
        + ", ".join(str(p) for p in PANEL_CANDIDATES)
        + ". Re-run the data acquisition cells or upload the cached panel first."
    )

required = ["ticker", "year", "ann_return_1y_fwd", "ann_return_3y_fwd"] + [f"pred_{x}" for x in LENS_NAMES]
missing = [c for c in required if c not in featured.columns]
if missing:
    raise RuntimeError(f"Featured panel is missing required columns: {missing}")

featured = featured.sort_values(["ticker", "year"]).reset_index(drop=True)
print("Featured:", featured.shape)
print("Tickers:", featured["ticker"].nunique())
print("Years:", int(featured["year"].min()), "-", int(featured["year"].max()))
print("Train rows:", len(featured[featured["year"] <= router.TRAIN_END_YEAR]))
print("Val rows:", len(featured[featured["year"] >= router.VAL_START_YEAR]))
display(featured[["ticker", "year", "omega_regime", "pred_reverseDcf", "pred_assetValue", "ann_return_3y_fwd"]].head())


## 5. Build Training Bundle


In [ ]:
bundle = build_omega_bundle(
    featured,
    max_years=10,
    train_end_year=router.TRAIN_END_YEAR,
    val_start_year=router.VAL_START_YEAR,
)

print("Train samples:", len(bundle.train))
print("Val samples:", len(bundle.val))
print("Feature count:", len(bundle.feature_cols))
print("Lens count:", len(bundle.lens_cols))
print("Regimes:", bundle.regimes)
print("Questions:", bundle.questions)

if len(bundle.train) == 0 or len(bundle.val) == 0:
    raise RuntimeError("Train/validation split produced empty data. Check years and forward-return columns.")


## 6. Train AURORA Omega


In [ ]:
stamp = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")
out_dir = ARTIFACT_ROOT / f"omega_colab_fixed_{stamp}"
out_dir.mkdir(parents=True, exist_ok=True)

cfg = TrainConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    d_model=D_MODEL,
    device=DEVICE,
    seed=SEED,
)

model, train_report = train_omega(bundle, cfg, out_dir)
eval_out = evaluate_omega(model, bundle, cfg)
memos = write_valuation_mri(bundle, eval_out, out_dir, limit=500)

print("Artifacts:", out_dir)
print("Train report:", train_report)
print("Eval metrics:", json.dumps(eval_out["metrics"], indent=2))
print("Memos:", len(memos))


## 7. Baselines and Diagnostics


In [ ]:
from sklearn.metrics import mean_absolute_error

val_frame = bundle.frame.iloc[eval_out["row_index"]].copy().reset_index(drop=True)
val_frame["omega_pred_1y"] = eval_out["pred_returns"][:, 0]
val_frame["omega_pred_3y"] = eval_out["pred_returns"][:, 1]
val_frame["omega_moe_3y"] = eval_out["omega_return"]

def _mae(pred, target):
    ok = np.isfinite(pred) & np.isfinite(target)
    return float(mean_absolute_error(np.asarray(target)[ok], np.asarray(pred)[ok])) if ok.any() else None

target_3y = val_frame["ann_return_3y_fwd"].to_numpy()
lens_mae = {
    name: _mae(val_frame[f"pred_{name}"].to_numpy(), target_3y)
    for name in LENS_NAMES
    if f"pred_{name}" in val_frame.columns
}
uniform_pred = val_frame[[f"pred_{name}" for name in LENS_NAMES]].mean(axis=1).to_numpy()

baseline = {
    "omega_direct_3y_mae": _mae(val_frame["omega_pred_3y"].to_numpy(), target_3y),
    "omega_moe_3y_mae": _mae(val_frame["omega_moe_3y"].to_numpy(), target_3y),
    "uniform_lens_3y_mae": _mae(uniform_pred, target_3y),
    "best_single_lens_3y": min(lens_mae.items(), key=lambda kv: kv[1] if kv[1] is not None else 999),
    "all_lens_mae": lens_mae,
}

with open(out_dir / "baseline_diagnostics.json", "w") as f:
    json.dump(baseline, f, indent=2)

print(json.dumps(baseline, indent=2))


## 8. Production Gate


In [ ]:
metrics = eval_out["metrics"]
max_weight = float(metrics.get("max_mean_lens_weight") or 1.0)
omega_mae = baseline["omega_moe_3y_mae"]
uniform_mae = baseline["uniform_lens_3y_mae"]
best_single_mae = baseline["best_single_lens_3y"][1]

gates = {
    "has_validation_rows": len(val_frame) >= 200,
    "not_collapsed": max_weight <= 0.45,
    "beats_uniform_3y_mae": omega_mae is not None and uniform_mae is not None and omega_mae < uniform_mae,
    "beats_best_single_3y_mae": omega_mae is not None and best_single_mae is not None and omega_mae < best_single_mae,
}
production_candidate = all(gates.values())

manifest = {
    "version": "aurora_omega_colab_fixed_v1",
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "artifact_dir": str(out_dir),
    "panel_rows": int(len(featured)),
    "panel_tickers": int(featured["ticker"].nunique()),
    "train_rows": int(len(bundle.train)),
    "val_rows": int(len(bundle.val)),
    "metrics": metrics,
    "baseline": baseline,
    "gates": gates,
    "production_candidate": production_candidate,
    "note": "Production means neural Omega beat the explicit baselines. If false, use as shadow research only.",
}

(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))


## 9. Inspect Latest Valuation MRI Memos


In [ ]:
mri_path = out_dir / "valuation_mri.jsonl"
print("MRI path:", mri_path)
if mri_path.exists():
    sample = [json.loads(line) for line in mri_path.read_text().splitlines()[:5]]
    display(pd.DataFrame(sample))
